# 🏀 March Machine Learning Mania 2026
## Stacked Ensemble — v2 (Improved)

### Key improvements over v1:
- ❌ **Removed bias correction** (was moving submission means further from 0.5)
- 🔧 **Tighter XGBoost** (max_depth 3→2, min_child_weight 3→5, stronger regularization)
- 🧠 **Women's: FGN replaced by VSN** (Variable Selection Network — better for small feature sets)
- 🎛️ **Stronger NN regularization** (dropout increased, weight decay 0.01→0.05, patience 25→15)
- ✨ **Seed interaction features** added: `SeedGap` and `IsUpsetter`
- 📊 **Non-linear meta-learner for Men's** (GradientBoosting instead of LogisticRegression)
- 🏀 **Regular season games added to training** (weighted 1/10 vs tournament games)

## 0 · Install & Imports

In [2]:
# ADD catboost to the pip install line
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "pandas", "numpy", "scipy",
    "scikit-learn", "xgboost", "catboost",  # ← add catboost
    "matplotlib", "seaborn"
], capture_output=True)

# ADD this import alongside xgboost
from catboost import CatBoostClassifier

import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mstats
from scipy.optimize import minimize_scalar

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import brier_score_loss
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import xgboost as xgb

warnings.filterwarnings("ignore")

SEED   = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print("✓ All imports successful")

PyTorch : 2.10.0+cpu
Device  : cpu
✓ All imports successful


## 1 · Configuration

In [3]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR       = "C:\\Users\\eduah\\Desktop\\Kaggle Competions\\March ML Mania\\"
SUBMISSION_IN  = "C:\\Users\\eduah\\Desktop\\Kaggle Competitions\\March ML Mania\\SampleSubmissionStage1.csv"
SUBMISSION_OUT = "C:\\Users\\eduah\\Desktop\\Kaggle Competitions\\March ML Mania\\submission_v2.csv"

# ── Season filters ─────────────────────────────────────────────────────────────
MIN_SEASON_M   = 2003
MIN_SEASON_W   = 2010
VAL_SEASONS    = [2023, 2024, 2025]
CURRENT_SEASON = 2026

# ── Preprocessing ──────────────────────────────────────────────────────────────
MIN_GAMES      = 10
WINSOR_LOW     = 0.01
WINSOR_HIGH    = 0.99
RECENCY_DECAY  = 0.85
MASSEY_SYSTEMS = ["POM", "SAG", "MOR", "RPI", "KPI", "DUN", "RTH"]

# ── Regular season training weight (vs tournament = 1.0) ──────────────────────
# Tournament games are 10x more predictive of March performance
REG_SEASON_WEIGHT = 0.1

# ── Stacking ───────────────────────────────────────────────────────────────────
N_FOLDS        = 5

# ── Neural network (tighter regularization vs v1) ──────────────────────────────
NN_HIDDEN_1    = 128
NN_HIDDEN_2    = 64
NN_DROP_1      = 0.50   # was 0.4
NN_DROP_2      = 0.40   # was 0.3
NN_DROP_RES    = 0.25   # was 0.2
NN_LR          = 3e-4
NN_WD          = 0.05   # was 0.01
NN_BATCH       = 64
NN_EPOCHS      = 300
NN_PATIENCE    = 15     # was 25 — force earlier stopping
NN_LABEL_SMOOTH= 0.07   # was 0.05

# ── Output ─────────────────────────────────────────────────────────────────────
PROB_MIN       = 0.05
PROB_MAX       = 0.95

# ── XGBoost (tighter — small training set) ─────────────────────────────────────
'''XGB_PARAMS = dict(
    n_estimators=100,        # was 200
    max_depth=2,             # was 3 — critical for small datasets
    learning_rate=0.05,
    subsample=0.7,           # was 0.8
    colsample_bytree=0.6,    # was 0.8
    min_child_weight=5,      # was 3
    reg_alpha=0.5,           # was 0.1
    reg_lambda=2.0,          # was 1.0
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=SEED,
    verbosity=0,
)'''
CATBOOST_PARAMS = dict(
    iterations=300,
    depth=4,
    learning_rate=0.05,
    l2_leaf_reg=5.0,
    random_strength=1.0,
    bagging_temperature=0.8,
    border_count=64,
    verbose=0,
    random_seed=SEED,
    eval_metric="Logloss",
)
print("✓ Configuration loaded")
print(f"  Data dir     : {os.path.abspath(DATA_DIR)}")
print(f"  Val seasons  : {VAL_SEASONS}")
print(f"  NN arch      : {NN_HIDDEN_1} → {NN_HIDDEN_2} → residual({NN_HIDDEN_2}) → 1")
print(f"  Reg season weight : {REG_SEASON_WEIGHT} (vs tournament=1.0)")

✓ Configuration loaded
  Data dir     : C:\Users\eduah\Desktop\Kaggle Competions\March ML Mania
  Val seasons  : [2023, 2024, 2025]
  NN arch      : 128 → 64 → residual(64) → 1
  Reg season weight : 0.1 (vs tournament=1.0)


## 2 · Load Raw CSV Files

In [ ]:
def read(fname):
    path = os.path.join(DATA_DIR, fname)
    df   = pd.read_csv(path)
    print(f"  ✓ {fname:<48s} {len(df):>8,} rows")
    return df

print("Loading CSV files …")
m_teams      = read("MTeams.csv")
w_teams      = read("WTeams.csv")
m_team_conf  = read("MTeamConferences.csv")
w_team_conf  = read("WTeamConferences.csv")
conferences  = read("Conferences.csv")
m_coaches    = read("MTeamCoaches.csv")
m_reg        = read("MRegularSeasonDetailedResults.csv")
w_reg        = read("WRegularSeasonDetailedResults.csv")
m_tourney    = read("MNCAATourneyCompactResults.csv")
w_tourney    = read("WNCAATourneyCompactResults.csv")
m_seeds      = read("MNCAATourneySeeds.csv")
w_seeds      = read("WNCAATourneySeeds.csv")
m_rankings   = read("MMasseyOrdinals.csv")
sample_sub   = read(SUBMISSION_IN)

print(f"\nSubmission rows to predict: {len(sample_sub):,}")

Loading CSV files …
  ✓ MTeams.csv                                            381 rows
  ✓ WTeams.csv                                            379 rows
  ✓ MTeamConferences.csv                               13,753 rows
  ✓ WTeamConferences.csv                                9,853 rows
  ✓ Conferences.csv                                        51 rows
  ✓ MTeamCoaches.csv                                   13,900 rows


## 3 · Preprocessing

In [ ]:
# ── 3.1  Master team identity ─────────────────────────────────────────────────
def build_master(m_team_conf, w_team_conf, m_teams, w_teams,
                 conferences, m_coaches):
    m = m_team_conf.merge(m_teams[["TeamID","TeamName"]], on="TeamID", how="left")
    m = m.merge(conferences, on="ConfAbbrev", how="left")
    m.rename(columns={"Description":"ConferenceName"}, inplace=True)
    m["IsWomens"] = 0

    w = w_team_conf.merge(w_teams[["TeamID","TeamName"]], on="TeamID", how="left")
    w = w.merge(conferences, on="ConfAbbrev", how="left")
    w.rename(columns={"Description":"ConferenceName"}, inplace=True)
    w["IsWomens"] = 1

    master = pd.concat([m, w], ignore_index=True)
    coach = (m_coaches.sort_values("LastDayNum", ascending=False)
                      .groupby(["Season","TeamID"]).first()
                      .reset_index()[["Season","TeamID","CoachName"]])
    master = master.merge(coach, on=["Season","TeamID"], how="left")
    master["CoachName"] = master["CoachName"].fillna("unknown")
    return master

master = build_master(m_team_conf, w_team_conf, m_teams, w_teams,
                      conferences, m_coaches)
print(f"Master table: {len(master):,} rows")

Master table: 23,606 rows


In [ ]:
# ── 3.2  Flatten game rows ────────────────────────────────────────────────────
def flatten_games(df):
    stat_cols = ["FGM","FGA","FGM3","FGA3","FTM","FTA",
                 "OR","DR","Ast","TO","Stl","Blk","PF"]
    w_ren = {"WTeamID":"TeamID","LTeamID":"OppID",
             "WScore":"Score","LScore":"OppScore"}
    for s in stat_cols:
        w_ren[f"W{s}"] = s
        w_ren[f"L{s}"] = f"Opp{s}"
    wins = df.rename(columns=w_ren).copy()
    wins["Win"]  = 1
    wins["Home"] = (wins["WLoc"] == "H").astype(int)

    l_ren = {"LTeamID":"TeamID","WTeamID":"OppID",
             "LScore":"Score","WScore":"OppScore"}
    for s in stat_cols:
        l_ren[f"L{s}"] = s
        l_ren[f"W{s}"] = f"Opp{s}"
    losses = df.rename(columns=l_ren).copy()
    losses["Win"]  = 0
    losses["Home"] = (losses["WLoc"] == "A").astype(int)

    keep = (["Season","DayNum","TeamID","OppID","Score","OppScore",
              "Win","Home","NumOT"] + stat_cols +
             [f"Opp{s}" for s in stat_cols])
    keep = [c for c in keep if c in wins.columns and c in losses.columns]
    return pd.concat([wins[keep], losses[keep]], ignore_index=True)


# ── 3.3  Aggregate per (Season, TeamID) ───────────────────────────────────────
def compute_stats(reg_df, min_season, label):
    df   = reg_df[reg_df["Season"] >= min_season].copy()
    flat = flatten_games(df)

    flat["Margin"] = flat["Score"] - flat["OppScore"]
    flat["Margin"] = mstats.winsorize(flat["Margin"].values,
                                      limits=[WINSOR_LOW, 1-WINSOR_HIGH])
    flat["Poss"]    = (flat["FGA"] - flat["OR"] + flat["TO"]
                       + 0.44*flat["FTA"]).clip(lower=1)
    flat["OppPoss"] = (flat["OppFGA"] - flat["OppOR"] + flat["OppTO"]
                       + 0.44*flat["OppFTA"]).clip(lower=1)
    flat["OffRtg"]  = flat["Score"]    / flat["Poss"]    * 100
    flat["DefRtg"]  = flat["OppScore"] / flat["OppPoss"] * 100
    flat["NetRtg"]  = flat["OffRtg"]   - flat["DefRtg"]

    agg = flat.groupby(["Season","TeamID"]).agg(
        Games=("Win","count"), Wins=("Win","sum"),
        AvgScore=("Score","mean"), AvgAllowed=("OppScore","mean"),
        AvgMargin=("Margin","mean"),
        FGM=("FGM","mean"), FGA=("FGA","mean"),
        FGM3=("FGM3","mean"), FGA3=("FGA3","mean"),
        FTM=("FTM","mean"), FTA=("FTA","mean"),
        OppFGM=("OppFGM","mean"), OppFGA=("OppFGA","mean"),
        AvgOR=("OR","mean"), AvgDR=("DR","mean"),
        AvgAst=("Ast","mean"), AvgTO=("TO","mean"),
        AvgStl=("Stl","mean"), AvgBlk=("Blk","mean"), AvgPF=("PF","mean"),
        AvgOffRtg=("OffRtg","mean"), AvgDefRtg=("DefRtg","mean"),
        AvgNetRtg=("NetRtg","mean"), HomeGames=("Home","sum"),
    ).reset_index()

    agg["WinRate"]    = agg["Wins"]   / agg["Games"]
    agg["FGPct"]      = agg["FGM"]    / agg["FGA"].replace(0,np.nan)
    agg["FG3Pct"]     = agg["FGM3"]   / agg["FGA3"].replace(0,np.nan)
    agg["FTPct"]      = agg["FTM"]    / agg["FTA"].replace(0,np.nan)
    agg["OppFGPct"]   = agg["OppFGM"] / agg["OppFGA"].replace(0,np.nan)
    agg["AstTORatio"] = agg["AvgAst"] / agg["AvgTO"].replace(0,np.nan)
    agg["FGM2"]       = agg["FGM"]    - agg["FGM3"]
    agg["FGA2"]       = agg["FGA"]    - agg["FGA3"]
    agg["FG2Pct"]     = agg["FGM2"]   / agg["FGA2"].replace(0,np.nan)

    late = flat[flat["DayNum"] >= 102].groupby(["Season","TeamID"]).agg(
        LateGames=("Win","count"), LateWinRate=("Win","mean"),
        LateAvgMargin=("Margin","mean"), LateAvgNetRtg=("NetRtg","mean"),
    ).reset_index()
    agg = agg.merge(late, on=["Season","TeamID"], how="left")

    before = len(agg)
    agg = agg[agg["Games"] >= MIN_GAMES].copy()
    print(f"  {label}: removed {before-len(agg)} team-seasons < {MIN_GAMES} games")

    num_cols = agg.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        if agg[col].isna().any():
            agg[col] = agg[col].fillna(agg[col].median())

    print(f"  {label}: {len(agg):,} team-season rows")
    return agg

print("Computing regular season stats …")
m_stats = compute_stats(m_reg, MIN_SEASON_M, "Men's")
w_stats = compute_stats(w_reg, MIN_SEASON_W, "Women's")

Computing regular season stats …
  Men's: removed 3 team-seasons < 10 games
  Men's: 8,343 team-season rows
  Women's: removed 10 team-seasons < 10 games
  Women's: 5,955 team-season rows


In [ ]:
# ── 3.4  Seeds ────────────────────────────────────────────────────────────────
def process_seeds(seeds_df):
    s = seeds_df.copy()
    s["SeedNum"] = s["Seed"].str[1:3].astype(int)
    return s[["Season","TeamID","SeedNum"]]

m_seeds_clean = process_seeds(m_seeds)
w_seeds_clean = process_seeds(w_seeds)
print(f"Seeds — Men's: {len(m_seeds_clean):,}  Women's: {len(w_seeds_clean):,}")

# ── 3.5  Massey rankings ──────────────────────────────────────────────────────
def process_massey(rankings_df):
    final   = rankings_df[rankings_df["RankingDayNum"] == 133].copy()
    avail   = [s for s in MASSEY_SYSTEMS if s in final["SystemName"].unique()]
    missing = [s for s in MASSEY_SYSTEMS if s not in final["SystemName"].unique()]
    if missing:
        print(f"  ⚠ Massey systems not found: {missing}")
    final = final[final["SystemName"].isin(avail)]
    piv = final.pivot_table(index=["Season","TeamID"],
                            columns="SystemName",
                            values="OrdinalRank",
                            aggfunc="mean").reset_index()
    piv.columns.name = None
    piv = piv.rename(columns={s: f"Rank_{s}" for s in avail})
    rank_cols = [f"Rank_{s}" for s in avail]
    for col in rank_cols:
        piv[col] = piv.groupby("Season")[col].transform(
            lambda x: x.fillna(x.median()))
    piv["AvgMasseyRank"] = piv[rank_cols].mean(axis=1)
    for col in rank_cols + ["AvgMasseyRank"]:
        piv[col] = piv[col].clip(upper=351)
    print(f"  Massey: {len(piv):,} team-seasons  systems={avail}")
    return piv

print("Processing Massey rankings …")
massey = process_massey(m_rankings)

Seeds — Men's: 2,694  Women's: 1,812
Processing Massey rankings …
  Massey: 7,992 team-seasons  systems=['POM', 'SAG', 'MOR', 'RPI', 'KPI', 'DUN', 'RTH']


In [ ]:
# ── 3.6  Assemble Team Season Profiles ───────────────────────────────────────
def build_profiles(master, m_stats, w_stats,
                   m_seeds_clean, w_seeds_clean, massey):
    all_stats = pd.concat([m_stats, w_stats], ignore_index=True)
    all_seeds = pd.concat([m_seeds_clean, w_seeds_clean], ignore_index=True)

    profiles = master.merge(all_stats, on=["Season","TeamID"], how="inner")
    profiles = profiles.merge(all_seeds, on=["Season","TeamID"], how="left")
    profiles["SeedNum"] = profiles["SeedNum"].fillna(17).astype(int)

    profiles = profiles.merge(massey, on=["Season","TeamID"], how="left")
    massey_cols = [c for c in profiles.columns
                   if c.startswith("Rank_") or c == "AvgMasseyRank"]
    for col in massey_cols:
        profiles[col] = profiles[col].fillna(200)

    max_s = profiles["Season"].max()
    profiles["RecencyWeight"] = RECENCY_DECAY ** (max_s - profiles["Season"])

    key_cols = ["WinRate","AvgMargin","FGPct","AvgNetRtg","SeedNum"]
    for col in key_cols:
        n = profiles[col].isna().sum()
        if n:
            profiles[col] = profiles[col].fillna(profiles[col].median())
            print(f"  ⚠ filled {n} NaN in {col}")

    print(f"Table A: {profiles.shape[0]:,} rows × {profiles.shape[1]} columns")
    print(f"  Seasons : {profiles['Season'].min()} – {profiles['Season'].max()}")
    print(f"  Men's   : {profiles['IsWomens'].eq(0).sum():,}")
    print(f"  Women's : {profiles['IsWomens'].eq(1).sum():,}")
    return profiles

profiles = build_profiles(master, m_stats, w_stats,
                          m_seeds_clean, w_seeds_clean, massey)

Table A: 14,298 rows × 54 columns
  Seasons : 2003 – 2026
  Men's   : 8,343
  Women's : 5,955


## 4 · Feature Engineering

**New in v2:** `SeedGap` (absolute seed difference) and `IsUpsetter` (seed says one thing, NetRtg says another) — both capture upset potential explicitly.

In [ ]:
FEAT_COLS_M = [
    # Tier 1 — strongest predictors
    "SeedNum_diff", "AvgNetRtg_diff", "AvgMasseyRank_diff", "WinRate_diff",
    # Tier 2 — form / momentum
    "LateWinRate_diff", "LateAvgMargin_diff", "LateAvgNetRtg_diff", "AvgMargin_diff",
    # Tier 3 — shooting efficiency
    "FGPct_diff", "FG3Pct_diff", "FTPct_diff", "FG2Pct_diff", "OppFGPct_diff",
    # Tier 4 — ball control
    "AstTORatio_diff", "AvgStl_diff", "AvgBlk_diff", "AvgTO_diff",
    # Tier 5 — rebounding
    "AvgOR_diff", "AvgDR_diff", "AvgAst_diff",
    # Tier 6 — individual ranking systems
    "Rank_POM_diff", "Rank_SAG_diff", "Rank_MOR_diff", "Rank_RPI_diff",
    "Rank_KPI_diff", "Rank_DUN_diff", "Rank_RTH_diff",
    # Tier 7 — NEW: interaction features
    "SeedGap", "IsUpsetter",
]

_massey_feats = [c for c in FEAT_COLS_M if "Rank_" in c or "Massey" in c]
FEAT_COLS_W   = [c for c in FEAT_COLS_M if c not in _massey_feats]

print(f"Men's features   : {len(FEAT_COLS_M)}")
print(f"Women's features : {len(FEAT_COLS_W)}")
print(f"New interaction features: SeedGap, IsUpsetter")

Men's features   : 29
Women's features : 21
New interaction features: SeedGap, IsUpsetter


In [ ]:
# ── 4.1  Build training matchups from TOURNAMENT games ────────────────────────
def build_tourney_matchups(m_tourney, w_tourney, profiles, feat_cols_m, feat_cols_w):
    """Tournament games — full recency weight (1.0 × season weight)."""
    tourney = pd.concat([
        m_tourney.assign(IsWomens=0),
        w_tourney.assign(IsWomens=1),
    ], ignore_index=True)

    rows, skipped = [], 0
    base_feat_cols_m = [c for c in feat_cols_m if c not in ["SeedGap","IsUpsetter"]]
    base_feat_cols_w = [c for c in feat_cols_w if c not in ["SeedGap","IsUpsetter"]]

    for _, game in tourney.iterrows():
        season    = game["Season"]
        t1        = min(game["WTeamID"], game["LTeamID"])
        t2        = max(game["WTeamID"], game["LTeamID"])
        label     = 1 if game["WTeamID"] == t1 else 0
        is_w      = int(game["IsWomens"])
        feat_cols = base_feat_cols_w if is_w else base_feat_cols_m

        p1 = profiles[(profiles["Season"]==season) & (profiles["TeamID"]==t1)]
        p2 = profiles[(profiles["Season"]==season) & (profiles["TeamID"]==t2)]
        if p1.empty or p2.empty:
            skipped += 1; continue
        p1, p2 = p1.iloc[0], p2.iloc[0]

        base_cols = [c.replace("_diff","") for c in feat_cols]
        row = {"Season":season,"T1":t1,"T2":t2,"Label":label,
               "IsWomens":is_w,"RecencyWeight":float(p1["RecencyWeight"])}
        for bc, dc in zip(base_cols, feat_cols):
            v1 = float(p1[bc]) if bc in p1.index else 0.0
            v2 = float(p2[bc]) if bc in p2.index else 0.0
            row[dc] = v1 - v2

        # Interaction features
        s1 = float(p1["SeedNum"]); s2 = float(p2["SeedNum"])
        n1 = float(p1["AvgNetRtg"]); n2 = float(p2["AvgNetRtg"])
        row["SeedGap"]    = abs(s1 - s2)
        row["IsUpsetter"] = float((s1 - s2) * (n1 - n2) < 0)  # seed & NetRtg disagree
        rows.append(row)

    matchups = pd.DataFrame(rows)
    print(f"Tournament matchups : {len(matchups):,}  (skipped {skipped})")
    print(f"Label balance       : {matchups['Label'].mean():.3f}")
    return matchups


# ── 4.2  Build training matchups from REGULAR SEASON games ───────────────────
def build_reg_season_matchups(m_reg, w_reg, profiles, feat_cols_m, feat_cols_w,
                               sample_frac=0.15):
    """
    Sample a fraction of regular season games to augment training.
    Weight = REG_SEASON_WEIGHT × season recency weight.
    sample_frac: fraction of regular season games to use (keep manageable).
    """
    base_feat_cols_m = [c for c in feat_cols_m if c not in ["SeedGap","IsUpsetter"]]
    base_feat_cols_w = [c for c in feat_cols_w if c not in ["SeedGap","IsUpsetter"]]

    def process_reg(reg_df, is_w, feat_cols, base_feat_cols):
        df = reg_df[reg_df["Season"] < CURRENT_SEASON].copy()
        df = df.sample(frac=sample_frac, random_state=SEED)
        rows = []
        for _, game in df.iterrows():
            season = game["Season"]
            t1 = min(game["WTeamID"], game["LTeamID"])
            t2 = max(game["WTeamID"], game["LTeamID"])
            label = 1 if game["WTeamID"] == t1 else 0

            p1 = profiles[(profiles["Season"]==season) & (profiles["TeamID"]==t1)]
            p2 = profiles[(profiles["Season"]==season) & (profiles["TeamID"]==t2)]
            if p1.empty or p2.empty: continue
            p1, p2 = p1.iloc[0], p2.iloc[0]

            row = {"Season":season,"T1":t1,"T2":t2,"Label":label,
                   "IsWomens":is_w,
                   "RecencyWeight": float(p1["RecencyWeight"]) * REG_SEASON_WEIGHT}
            for bc, dc in zip([c.replace("_diff","") for c in base_feat_cols], base_feat_cols):
                v1 = float(p1[bc]) if bc in p1.index else 0.0
                v2 = float(p2[bc]) if bc in p2.index else 0.0
                row[dc] = v1 - v2

            s1 = float(p1["SeedNum"]); s2 = float(p2["SeedNum"])
            n1 = float(p1["AvgNetRtg"]); n2 = float(p2["AvgNetRtg"])
            row["SeedGap"]    = abs(s1 - s2)
            row["IsUpsetter"] = float((s1 - s2) * (n1 - n2) < 0)
            rows.append(row)
        return pd.DataFrame(rows)

    reg_m = process_reg(m_reg, 0, FEAT_COLS_M, base_feat_cols_m)
    reg_w = process_reg(w_reg, 1, FEAT_COLS_W, base_feat_cols_w)
    reg_all = pd.concat([reg_m, reg_w], ignore_index=True)
    print(f"Regular season matchups: {len(reg_all):,}  "
          f"(weight={REG_SEASON_WEIGHT} × recency)")
    return reg_all


print("Building tournament matchups …")
tourney_matchups = build_tourney_matchups(
    m_tourney, w_tourney, profiles, FEAT_COLS_M, FEAT_COLS_W)

print("\nBuilding regular season matchups (15% sample) …")
reg_matchups = build_reg_season_matchups(
    m_reg, w_reg, profiles, FEAT_COLS_M, FEAT_COLS_W, sample_frac=0.15)

# Combine — tourney games come first (important for KFold reproducibility)
matchups = pd.concat([tourney_matchups, reg_matchups], ignore_index=True)
print(f"\nCombined training set: {len(matchups):,} rows")
print(f"  Tournament : {len(tourney_matchups):,}")
print(f"  Reg season : {len(reg_matchups):,}")
print(f"  Label balance: {matchups['Label'].mean():.3f}")

Building tournament matchups …
Tournament matchups : 2,410  (skipped 1892)
Label balance       : 0.502

Building regular season matchups (15% sample) …
Regular season matchups: 30,080  (weight=0.1 × recency)

Combined training set: 32,490 rows
  Tournament : 2,410
  Reg season : 30,080
  Label balance: 0.498


In [ ]:
# ── 4.3  Build submission features ───────────────────────────────────────────
def get_profile(team_id, profiles, season):
    row = profiles[(profiles["TeamID"]==team_id) & (profiles["Season"]==season)]
    if not row.empty: return row.iloc[0]
    row = profiles[(profiles["TeamID"]==team_id) &
                   (profiles["Season"]<=season)].sort_values("Season",ascending=False)
    if not row.empty: return row.iloc[0]
    row = profiles[profiles["TeamID"]==team_id].sort_values("Season",ascending=False)
    return row.iloc[0] if not row.empty else None

def build_sub_features(sample_sub, profiles, feat_cols_m):
    base_feat_cols_m = [c for c in feat_cols_m if c not in ["SeedGap","IsUpsetter"]]
    base_cols = [c.replace("_diff","") for c in base_feat_cols_m]
    rows, no_data = [], 0
    for _, row in sample_sub.iterrows():
        parts  = row["ID"].split("_")
        season = int(parts[0]); t1 = int(parts[1]); t2 = int(parts[2])
        is_w   = int(t1 >= 3000)
        p1 = get_profile(t1, profiles, season)
        p2 = get_profile(t2, profiles, season)
        feat_row = {"ID":row["ID"],"Season":season,"IsWomens":is_w}
        if p1 is None or p2 is None:
            no_data += 1
            for col in feat_cols_m: feat_row[col] = 0.0
        else:
            for bc, dc in zip(base_cols, base_feat_cols_m):
                v1 = float(p1[bc]) if bc in p1.index else np.nan
                v2 = float(p2[bc]) if bc in p2.index else np.nan
                feat_row[dc] = 0.0 if (pd.isna(v1) or pd.isna(v2)) else v1-v2
            s1 = float(p1["SeedNum"]); s2 = float(p2["SeedNum"])
            n1 = float(p1["AvgNetRtg"]); n2 = float(p2["AvgNetRtg"])
            feat_row["SeedGap"]    = abs(s1 - s2)
            feat_row["IsUpsetter"] = float((s1 - s2) * (n1 - n2) < 0)
        rows.append(feat_row)

    sub_feat = pd.DataFrame(rows)
    if no_data: print(f"  ⚠ {no_data} matchups missing profile → set to 0")
    print(f"Sub features  : {len(sub_feat):,} rows")
    print(f"  Men's       : {(sub_feat['IsWomens']==0).sum():,}")
    print(f"  Women's     : {(sub_feat['IsWomens']==1).sum():,}")
    return sub_feat

print("Building submission feature matrix …")
sub_features = build_sub_features(sample_sub, profiles, FEAT_COLS_M)

Building submission feature matrix …
Sub features  : 132,133 rows
  Men's       : 66,430
  Women's     : 65,703


In [ ]:
# ── 4.4  Train / validation split ────────────────────────────────────────────
# NOTE: Only use TOURNAMENT matchups for validation (reg season is augmentation)
def split(df, tourney_df, feat_cols):
    # Validation: only tourney games from VAL_SEASONS
    val_idx   = tourney_df[tourney_df["Season"].isin(VAL_SEASONS)].index
    train_idx = df.index.difference(val_idx)

    train = df.loc[train_idx]
    val   = df.loc[val_idx]

    X_tr = train[feat_cols].fillna(0).values.astype(np.float32)
    y_tr = train["Label"].values.astype(np.float32)
    w_tr = train["RecencyWeight"].values.astype(np.float32)
    X_vl = val[feat_cols].fillna(0).values.astype(np.float32)
    y_vl = val["Label"].values.astype(np.float32)
    return X_tr, y_tr, w_tr, X_vl, y_vl

m_matchups       = matchups[matchups["IsWomens"]==0].copy()
w_matchups       = matchups[matchups["IsWomens"]==1].copy()
m_tourney_subset = tourney_matchups[tourney_matchups["IsWomens"]==0]
w_tourney_subset = tourney_matchups[tourney_matchups["IsWomens"]==1]

X_train_m, y_train_m, w_train_m, X_val_m, y_val_m = split(
    m_matchups, m_tourney_subset, FEAT_COLS_M)
X_train_w, y_train_w, w_train_w, X_val_w, y_val_w = split(
    w_matchups, w_tourney_subset, FEAT_COLS_W)

print(f"Men's   → Train: {len(X_train_m):,}  Val: {len(X_val_m):,}  Features: {len(FEAT_COLS_M)}")
print(f"Women's → Train: {len(X_train_w):,}  Val: {len(X_val_w):,}  Features: {len(FEAT_COLS_W)}")
print(f"Label balance — Men's train: {y_train_m.mean():.3f}  Women's train: {y_train_w.mean():.3f}")

Men's   → Train: 19,080  Val: 201  Features: 29
Women's → Train: 13,008  Val: 201  Features: 21
Label balance — Men's train: 0.489  Women's train: 0.510


## 5 · Model Architectures

In [ ]:
# ── 5.1  Baseline MLP with residual connection ────────────────────────────────
class MarchMadnessNet(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.input_bn = nn.BatchNorm1d(input_dim)
        self.block1 = nn.Sequential(
            nn.Linear(input_dim, NN_HIDDEN_1),
            nn.BatchNorm1d(NN_HIDDEN_1), nn.GELU(), nn.Dropout(NN_DROP_1),
        )
        self.block2 = nn.Sequential(
            nn.Linear(NN_HIDDEN_1, NN_HIDDEN_2),
            nn.BatchNorm1d(NN_HIDDEN_2), nn.GELU(), nn.Dropout(NN_DROP_2),
        )
        self.block3 = nn.Sequential(
            nn.Linear(NN_HIDDEN_2, NN_HIDDEN_2),
            nn.BatchNorm1d(NN_HIDDEN_2), nn.GELU(), nn.Dropout(NN_DROP_RES),
        )
        self.output = nn.Sequential(nn.Linear(NN_HIDDEN_2, 1), nn.Sigmoid())
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.input_bn(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x) + x
        return self.output(x)


# ── 5.2  Variable Selection Network (VSN) ────────────────────────────────────
# Learns per-sample feature importance — replaces FGN for Women's
# A #1 vs #16 matchup weights seed heavily; a #5 vs #4 focuses on form.
class VariableSelectionNetwork(nn.Module):
    def __init__(self, input_dim: int, gate_hidden: int = 64,
                 enc_hidden_1: int = 128, enc_hidden_2: int = 64,
                 dropout: float = 0.3):
        super().__init__()
        self.gate = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, gate_hidden),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(gate_hidden, input_dim),
            nn.Softmax(dim=-1),
        )
        self.encoder = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, enc_hidden_1),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(enc_hidden_1, enc_hidden_2),
            nn.GELU(),
            nn.Dropout(dropout * 0.75),
            nn.Linear(enc_hidden_2, enc_hidden_2),
        )
        self.residual_proj = nn.Linear(input_dim, enc_hidden_2)
        self.res_bn  = nn.BatchNorm1d(enc_hidden_2)
        self.res_act = nn.GELU()
        self.res_drop = nn.Dropout(dropout * 0.5)
        self.output  = nn.Sequential(nn.Linear(enc_hidden_2, 1), nn.Sigmoid())
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        w   = self.gate(x)
        x_w = x * w * x.shape[1]
        h   = self.encoder(x_w)
        res = self.residual_proj(x_w)
        h   = self.res_drop(self.res_act(self.res_bn(h + res)))
        return self.output(h)

    def get_feature_weights(self, x):
        self.eval()
        with torch.no_grad():
            return self.gate(x)


# ── 5.3  FeatureGroupNet — Men's only (3 groups: rankings, efficiency, form) ──
class FeatureGroupNet(nn.Module):
    def __init__(self, feature_groups: dict, embed_dim: int = 32,
                 fusion_hidden: int = 64, dropout: float = 0.3):
        super().__init__()
        self.groups = feature_groups
        self.encoders = nn.ModuleDict()
        for name, indices in feature_groups.items():
            self.encoders[name] = nn.Sequential(
                nn.BatchNorm1d(len(indices)),
                nn.Linear(len(indices), embed_dim), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(embed_dim, embed_dim),    nn.GELU(), nn.Dropout(dropout*0.75),
            )
        fused_dim = embed_dim * len(feature_groups)
        self.fusion = nn.Sequential(
            nn.BatchNorm1d(fused_dim),
            nn.Linear(fused_dim, fusion_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(fusion_hidden, fusion_hidden//2), nn.GELU(), nn.Dropout(dropout*0.5),
            nn.Linear(fusion_hidden//2, 1), nn.Sigmoid(),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        embeddings = [self.encoders[n](x[:, idx]) for n, idx in self.groups.items()]
        return self.fusion(torch.cat(embeddings, dim=1))


# ── Feature group index maps (Men's only) ─────────────────────────────────────
FEATURE_GROUPS_M = {
    "rankings"  : ["SeedNum_diff","AvgMasseyRank_diff","Rank_POM_diff","Rank_SAG_diff",
                   "Rank_MOR_diff","Rank_RPI_diff","Rank_KPI_diff","Rank_DUN_diff",
                   "Rank_RTH_diff","SeedGap","IsUpsetter"],
    "efficiency": ["AvgNetRtg_diff","WinRate_diff","AvgMargin_diff","FGPct_diff",
                   "FG3Pct_diff","FTPct_diff","FG2Pct_diff","OppFGPct_diff"],
    "form"      : ["LateWinRate_diff","LateAvgMargin_diff","LateAvgNetRtg_diff",
                   "AstTORatio_diff","AvgStl_diff","AvgBlk_diff",
                   "AvgTO_diff","AvgOR_diff","AvgDR_diff","AvgAst_diff"],
}

def make_idx_groups(feature_groups_dict, feat_cols):
    feat_idx = {f: i for i, f in enumerate(feat_cols)}
    return {name: [feat_idx[f] for f in feats if f in feat_idx]
            for name, feats in feature_groups_dict.items()
            if any(f in feat_idx for f in feats)}

idx_groups_m = make_idx_groups(FEATURE_GROUPS_M, FEAT_COLS_M)

# Shape checks
_x_m = torch.randn(8, len(FEAT_COLS_M))
_x_w = torch.randn(8, len(FEAT_COLS_W))
assert MarchMadnessNet(len(FEAT_COLS_M))(_x_m).shape == (8, 1)
assert VariableSelectionNetwork(len(FEAT_COLS_W))(_x_w).shape == (8, 1)
assert FeatureGroupNet(idx_groups_m)(_x_m).shape == (8, 1)
print("✓ All model architectures verified")
print(f"  Men's FGN groups : {list(idx_groups_m.keys())}")

✓ All model architectures verified
  Men's FGN groups : ['rankings', 'efficiency', 'form']


## 6 · OOF & Retrain Helpers

In [ ]:
def get_oof_sklearn(model_fn, X, y, w, label="LR"):
    oof = np.zeros(len(y))
    kf  = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_scores = []
    for fold, (tr, vl) in enumerate(kf.split(X)):
        sc  = StandardScaler()
        Xtr = sc.fit_transform(X[tr])
        Xvl = sc.transform(X[vl])
        m   = model_fn()
        m.fit(Xtr, y[tr], sample_weight=w[tr])
        oof[vl] = m.predict_proba(Xvl)[:, 1]
        s = brier_score_loss(y[vl], oof[vl])
        fold_scores.append(s)
        print(f"    [{label}] fold {fold+1}/{N_FOLDS}  brier={s:.4f}")
    overall = brier_score_loss(y, oof)
    print(f"  → {label} OOF brier: {overall:.4f}  std={np.std(fold_scores):.4f}")
    return oof, overall

# REMOVE get_oof_xgboost() and retrain_xgb() entirely.
# REPLACE with these two functions:

def get_oof_catboost(X, y, w, label="CAT"):
    oof = np.zeros(len(y))
    kf  = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_scores = []
    for fold, (tr, vl) in enumerate(kf.split(X)):
        m = CatBoostClassifier(**CATBOOST_PARAMS)
        m.fit(X[tr], y[tr], sample_weight=w[tr])
        oof[vl] = m.predict_proba(X[vl])[:, 1]
        s = brier_score_loss(y[vl], oof[vl])
        fold_scores.append(s)
        print(f"    [{label}] fold {fold+1}/{N_FOLDS}  brier={s:.4f}")
    overall = brier_score_loss(y, oof)
    print(f"  → {label} OOF brier: {overall:.4f}  std={np.std(fold_scores):.4f}")
    return oof, overall


def retrain_catboost(X, y, w):
    m = CatBoostClassifier(**CATBOOST_PARAMS)
    m.fit(X, y, sample_weight=w)
    return m


def get_oof_nn(model_class, X, y, w, feat_dim, label="NN"):
    oof = np.zeros(len(y))
    kf  = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_scores = []
    bce = nn.BCELoss(reduction="none")

    for fold, (tr, vl) in enumerate(kf.split(X)):
        sc   = StandardScaler()
        Xtr  = sc.fit_transform(X[tr]).astype(np.float32)
        Xvl  = sc.transform(X[vl]).astype(np.float32)
        y_tr, y_vl, w_tr = y[tr], y[vl], w[tr]

        ds = TensorDataset(torch.tensor(Xtr), torch.tensor(y_tr), torch.tensor(w_tr))
        dl = DataLoader(ds, batch_size=NN_BATCH, shuffle=True)

        try:
            model = model_class(input_dim=feat_dim).to(DEVICE)
        except TypeError:
            model = model_class(feat_dim).to(DEVICE)

        opt   = optim.AdamW(model.parameters(), lr=NN_LR, weight_decay=NN_WD)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NN_EPOCHS, eta_min=1e-6)

        best_val, best_w, no_imp = np.inf, None, 0
        for epoch in range(1, NN_EPOCHS+1):
            model.train()
            for Xb, yb, wb in dl:
                Xb, yb, wb = Xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
                opt.zero_grad()
                pred = model(Xb).squeeze()
                yb_s = yb*(1-NN_LABEL_SMOOTH) + (1-yb)*NN_LABEL_SMOOTH
                loss = (bce(pred, yb_s)*wb).mean()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            model.eval()
            with torch.no_grad():
                p = model(torch.tensor(Xvl).to(DEVICE)).squeeze().cpu().numpy()
            vb = brier_score_loss(y_vl, np.clip(p, PROB_MIN, PROB_MAX))
            if vb < best_val - 1e-5:
                best_val = vb
                best_w   = {k: v.clone() for k, v in model.state_dict().items()}
                no_imp   = 0
            else:
                no_imp += 1
                if no_imp >= NN_PATIENCE: break

        model.load_state_dict(best_w)
        model.eval()
        with torch.no_grad():
            p = model(torch.tensor(Xvl).to(DEVICE)).squeeze().cpu().numpy()
        oof[vl] = np.clip(p, PROB_MIN, PROB_MAX)
        s = brier_score_loss(y_vl, oof[vl])
        fold_scores.append(s)
        print(f"    [{label}] fold {fold+1}/{N_FOLDS}  "
              f"stopped≈ep{NN_EPOCHS-no_imp}  best_val={best_val:.4f}  oof={s:.4f}")

    overall = brier_score_loss(y, oof)
    print(f"  → {label} OOF brier: {overall:.4f}  std={np.std(fold_scores):.4f}")
    return oof, overall


print("✓ OOF helpers defined")

✓ OOF helpers defined


In [ ]:
# ── Temperature scaling ───────────────────────────────────────────────────────
def temperature_scale(probs, T):
    probs  = np.clip(probs, 1e-7, 1 - 1e-7)
    logits = np.log(probs / (1 - probs))
    return 1 / (1 + np.exp(-logits / T))

def find_optimal_temperature(raw_probs, y_true, label=""):
    result = minimize_scalar(
        lambda T: brier_score_loss(
            y_true, np.clip(temperature_scale(raw_probs, T), PROB_MIN, PROB_MAX)
        ),
        bounds=(0.1, 5.0), method="bounded"
    )
    T_opt    = result.x
    original = brier_score_loss(y_true, np.clip(raw_probs, PROB_MIN, PROB_MAX))
    improved = brier_score_loss(y_true,
                  np.clip(temperature_scale(raw_probs, T_opt), PROB_MIN, PROB_MAX))
    print(f"  {label} optimal T={T_opt:.4f}  "
          f"brier: {original:.4f} → {improved:.4f}  "
          f"({'more' if T_opt < 1 else 'less'} confident)")
    return T_opt

print("✓ Temperature scaling defined")

✓ Temperature scaling defined


In [ ]:
# ── Retrain helpers ───────────────────────────────────────────────────────────
def retrain_lr(X, y, w):
    sc  = StandardScaler()
    Xsc = sc.fit_transform(X)
    raw = LogisticRegression(C=0.1, max_iter=2000, random_state=SEED, solver="lbfgs")
    raw.fit(Xsc, y, sample_weight=w)
    cal = CalibratedClassifierCV(raw, cv="prefit", method="sigmoid")
    cal.fit(Xsc, y)
    return cal, sc

def retrain_xgb(X, y, w):
    m   = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(X, y, sample_weight=w)
    cal = CalibratedClassifierCV(m, cv="prefit", method="sigmoid")
    cal.fit(X, y)
    return cal

def retrain_nn(model_class, X, y, w, feat_dim):
    sp   = int(len(X)*0.85)
    sc   = StandardScaler()
    Xtr  = sc.fit_transform(X[:sp]).astype(np.float32)
    Xvl  = sc.transform(X[sp:]).astype(np.float32)
    y_tr, y_vl, w_tr = y[:sp], y[sp:], w[:sp]

    ds   = TensorDataset(torch.tensor(Xtr), torch.tensor(y_tr), torch.tensor(w_tr))
    dl   = DataLoader(ds, batch_size=NN_BATCH, shuffle=True)
    bce  = nn.BCELoss(reduction="none")

    try:
        model = model_class(input_dim=feat_dim).to(DEVICE)
    except TypeError:
        model = model_class(feat_dim).to(DEVICE)

    opt   = optim.AdamW(model.parameters(), lr=NN_LR, weight_decay=NN_WD)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NN_EPOCHS, eta_min=1e-6)

    best_val, best_w, no_imp = np.inf, None, 0
    for epoch in range(1, NN_EPOCHS+1):
        model.train()
        for Xb, yb, wb in dl:
            Xb, yb, wb = Xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
            opt.zero_grad()
            pred = model(Xb).squeeze()
            yb_s = yb*(1-NN_LABEL_SMOOTH) + (1-yb)*NN_LABEL_SMOOTH
            loss = (bce(pred, yb_s)*wb).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            p = model(torch.tensor(Xvl).to(DEVICE)).squeeze().cpu().numpy()
        vb = brier_score_loss(y_vl, np.clip(p, PROB_MIN, PROB_MAX))
        if vb < best_val - 1e-5:
            best_val = vb
            best_w   = {k: v.clone() for k, v in model.state_dict().items()}
            no_imp   = 0
        else:
            no_imp += 1
            if no_imp >= NN_PATIENCE: break

    model.load_state_dict(best_w)
    print(f"    NN retrain best val brier: {best_val:.4f}")
    return model, sc


def predict_nn(model, scaler, X, bs=4096):
    Xsc = scaler.transform(X).astype(np.float32)
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(Xsc), bs):
            b = torch.tensor(Xsc[i:i+bs]).to(DEVICE)
            p = model(b).squeeze().cpu().numpy()
            preds.extend(p if p.ndim > 0 else [float(p)])
    return np.clip(np.array(preds), PROB_MIN, PROB_MAX)


print("✓ Retrain helpers defined")

✓ Retrain helpers defined


## 7 · Meta-Learner

**v2 change:** Men's uses `GradientBoostingClassifier` (non-linear, captures model interactions). Women's keeps `LogisticRegression` (fewer base models, simpler is safer).

In [ ]:
class MetaLearner:
    def __init__(self, use_gbm=False):
        """
        use_gbm=True  : GradientBoosting meta-learner (Men's — 4 models, more data)
        use_gbm=False : LogisticRegression meta-learner (Women's — 3 models)
        """
        if use_gbm:
            self.model = GradientBoostingClassifier(
                n_estimators=50, max_depth=2, learning_rate=0.1,
                subsample=0.8, random_state=SEED
            )
        else:
            self.model = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
        self.scaler  = StandardScaler()
        self.use_gbm = use_gbm

    def fit(self, oof_matrix, y, names=None):
        self.names = names or [f"M{i+1}" for i in range(oof_matrix.shape[1])]
        Xsc = self.scaler.fit_transform(oof_matrix)
        self.model.fit(Xsc, y)
        blend = np.clip(self.model.predict_proba(Xsc)[:,1], PROB_MIN, PROB_MAX)
        print(f"  OOF blend brier (indicative): {brier_score_loss(y, blend):.4f}")
        if not self.use_gbm:
            print(f"  Coefficients:")
            for name, coef in zip(self.names, self.model.coef_[0]):
                bar = "█" * max(1, int(abs(coef)*30))
                print(f"    {name:<10s}: {coef:+.4f}  {bar}")
        else:
            print(f"  Feature importances (GBM):")
            for name, imp in zip(self.names, self.model.feature_importances_):
                bar = "█" * max(1, int(imp*100))
                print(f"    {name:<10s}: {imp:.4f}  {bar}")

    def predict_proba(self, pred_matrix):
        return self.model.predict_proba(
            self.scaler.transform(pred_matrix))[:,1]

print("✓ MetaLearner defined (LR for Women's, GBM for Men's)")

✓ MetaLearner defined (LR for Women's, GBM for Men's)


## 8 · Full Stacking Pipeline

In [ ]:
def run_stacking(X_train, y_train, w_train,
                 X_val,   y_val,
                 X_sub,
                 feat_dim,
                 nn_models,
                 use_gbm_meta=False,
                 gender_label="Men's"):

    history     = {}
    model_names = ["LR", "CAT"] + [name for name, _ in nn_models]

    # ── Phase 1: OOF ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  {gender_label}  |  Phase 1 — OOF ({N_FOLDS}-fold CV, {len(model_names)} models)")
    print(f"{'='*60}")

    print("\n  [1] Logistic Regression")
    oof_lr, _ = get_oof_sklearn(
        lambda: LogisticRegression(C=0.1, max_iter=2000,
                                   random_state=SEED, solver="lbfgs"),
        X_train, y_train, w_train, label="LR")

    print("\n  [2] CatBoost")
    oof_cat, _ = get_oof_catboost(X_train, y_train, w_train, label="CAT")
    T_cat = find_optimal_temperature(oof_cat, y_train,
                                     label=f"{gender_label} CAT (OOF T-scale)")
    oof_cat_scaled = temperature_scale(oof_cat, T_cat)
    history["T_cat"] = T_cat

    oof_nns = []
    for i, (name, model_fn) in enumerate(nn_models):
        print(f"\n  [{3+i}] {name}")
        oof_nn, _ = get_oof_nn(
            model_fn, X_train, y_train, w_train,
            feat_dim=feat_dim, label=name)
        oof_nns.append((name, oof_nn))

    # ── Phase 2: Meta-learner ─────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  {gender_label}  |  Phase 2 — Meta-learner ({'GBM' if use_gbm_meta else 'LR'})")
    print(f"{'='*60}")

    oof_parts = [oof_lr, oof_cat_scaled] + [oof for _, oof in oof_nns]
    oof_stack = np.column_stack(oof_parts)
    meta = MetaLearner(use_gbm=use_gbm_meta)
    meta.fit(oof_stack, y_train, names=model_names)

    # ── Phase 3: Retrain on full data ─────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  {gender_label}  |  Phase 3 — Retraining on 100% training data")
    print(f"{'='*60}")

    print("  Retraining LR …")
    lr_full, sc_lr_full = retrain_lr(X_train, y_train, w_train)
    print("  Retraining CatBoost …")
    cat_full = retrain_catboost(X_train, y_train, w_train)

    nn_fulls = []
    for name, model_fn in nn_models:
        print(f"  Retraining {name} …")
        nn_full, sc_nn_full = retrain_nn(model_fn, X_train, y_train, w_train, feat_dim)
        nn_fulls.append((name, nn_full, sc_nn_full))

    # ── Phase 4: Validation ───────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  {gender_label}  |  Phase 4 — Validation Brier scores")
    print(f"{'='*60}")

    val_lr  = lr_full.predict_proba(sc_lr_full.transform(X_val))[:,1]
    val_cat_raw = cat_full.predict_proba(X_val)[:,1]
    val_cat = np.clip(temperature_scale(val_cat_raw, T_cat), PROB_MIN, PROB_MAX)
    

    val_nns = []
    for name, nn_full, sc_nn_full in nn_fulls:
        val_nns.append((name, predict_nn(nn_full, sc_nn_full, X_val)))

    val_parts = [val_lr, val_cat] + [p for _, p in val_nns]
    val_avg   = np.clip(np.mean(val_parts, axis=0), PROB_MIN, PROB_MAX)
    val_stack = np.column_stack(val_parts)
    meta_val  = np.clip(meta.predict_proba(val_stack), PROB_MIN, PROB_MAX)

    base = brier_score_loss(y_val, np.full(len(y_val), 0.5))
    history["Baseline"] = base

    print(f"\n  {'Model':<22s}  {'Brier':>7}  {'vs baseline':>12}")
    print(f"  {'─'*46}")
    for name, preds in (
        [("Baseline (0.5)", np.full(len(y_val), 0.5)),
         ("LR",              val_lr),
         ("CB (T-scaled)",  val_cat)]
        + [(n, p) for n, p in val_nns]
        + [("Simple average", val_avg),
           ("★ Meta stack",   meta_val)]
    ):
        b   = brier_score_loss(y_val, np.clip(preds, PROB_MIN, PROB_MAX))
        imp = (base - b)/base*100
        mrk = "  ←" if name == "★ Meta stack" else ""
        print(f"  {name:<22s}  {b:>7.4f}  {imp:>+11.1f}%{mrk}")
        history[f"{name}_val"] = b

    # Store val predictions for calibration plots
    base_preds = {"LR": val_lr, "CAT": val_cat, "Avg": val_avg}
    for n, p in val_nns: base_preds[n] = p

    # ── Phase 5: Submission predictions ───────────────────────────────────────
    sub_lr  = lr_full.predict_proba(sc_lr_full.transform(X_sub))[:,1]
    sub_cat = np.clip(temperature_scale(
              cat_full.predict_proba(X_sub)[:,1], T_cat), PROB_MIN, PROB_MAX)
    sub_parts = [sub_lr, sub_cat]
    for name, nn_full, sc_nn_full in nn_fulls:
        sub_parts.append(predict_nn(nn_full, sc_nn_full, X_sub))

    meta_sub = np.clip(
        meta.predict_proba(np.column_stack(sub_parts)), PROB_MIN, PROB_MAX)

    history["nn_fulls"] = nn_fulls
    return meta_val, meta_sub, meta, history, base_preds


print("✓ run_stacking defined")

✓ run_stacking defined


## 9 · Run — Men's

Base models: LR + XGBoost (tighter) + MarchMadnessNet + FeatureGroupNet  
Meta: GradientBoosting (non-linear)

In [ ]:
X_sub_m = (sub_features[sub_features["IsWomens"]==0][FEAT_COLS_M]
           .fillna(0).values.astype(np.float32))

val_meta_m, sub_meta_m, meta_m, hist_m, base_preds_m = run_stacking(
    X_train        = X_train_m,
    y_train        = y_train_m,
    w_train        = w_train_m,
    X_val          = X_val_m,
    y_val          = y_val_m,
    X_sub          = X_sub_m,
    feat_dim       = len(FEAT_COLS_M),
    nn_models      = [
        ("NN",  MarchMadnessNet),
        ("VSN", VariableSelectionNetwork),
    ],
    use_gbm_meta   = True,   # Non-linear meta for Men's
    gender_label   = "Men's",
)


  Men's  |  Phase 1 — OOF (5-fold CV, 4 models)

  [1] Logistic Regression
    [LR] fold 1/5  brier=0.1628
    [LR] fold 2/5  brier=0.1688
    [LR] fold 3/5  brier=0.1675
    [LR] fold 4/5  brier=0.1679
    [LR] fold 5/5  brier=0.1632
  → LR OOF brier: 0.1660  std=0.0025

  [2] CatBoost
    [CAT] fold 1/5  brier=0.1623
    [CAT] fold 2/5  brier=0.1699
    [CAT] fold 3/5  brier=0.1682
    [CAT] fold 4/5  brier=0.1693
    [CAT] fold 5/5  brier=0.1637
  → CAT OOF brier: 0.1667  std=0.0031
  Men's CAT (OOF T-scale) optimal T=1.0731  brier: 0.1668 → 0.1666  (less confident)

  [3] NN
    [NN] fold 1/5  stopped≈ep285  best_val=0.1653  oof=0.1653
    [NN] fold 2/5  stopped≈ep285  best_val=0.1708  oof=0.1708
    [NN] fold 3/5  stopped≈ep285  best_val=0.1685  oof=0.1685
    [NN] fold 4/5  stopped≈ep285  best_val=0.1694  oof=0.1694
    [NN] fold 5/5  stopped≈ep285  best_val=0.1656  oof=0.1656
  → NN OOF brier: 0.1679  std=0.0022

  [4] VSN
    [VSN] fold 1/5  stopped≈ep285  best_val=0.1654  oof

## 10 · Run — Women's

Base models: LR + XGBoost (tighter) + MarchMadnessNet + **VSN** (replaces FGN)  
Meta: LogisticRegression (3 models, simpler is safer)

In [ ]:
X_sub_w = (sub_features[sub_features["IsWomens"]==1][FEAT_COLS_W]
           .fillna(0).values.astype(np.float32))

val_meta_w, sub_meta_w, meta_w, hist_w, base_preds_w = run_stacking(
    X_train        = X_train_w,
    y_train        = y_train_w,
    w_train        = w_train_w,
    X_val          = X_val_w,
    y_val          = y_val_w,
    X_sub          = X_sub_w,
    feat_dim       = len(FEAT_COLS_W),
    nn_models      = [
        ("NN",  MarchMadnessNet),
        ("VSN", VariableSelectionNetwork),  # replaces FGN for Women's
    ],
    use_gbm_meta   = False,  # LR meta for Women's
    gender_label   = "Women's",
)

## 11 · Visualisations

In [ ]:
# ── 11.1  Validation Brier bar chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, hist, label in [(axes[0], hist_m, "Men's"), (axes[1], hist_w, "Women's")]:
    val_keys = {k: v for k, v in hist.items() if k.endswith("_val")}
    names  = [k.replace("_val","") for k in val_keys]
    scores = list(val_keys.values())
    colors = plt.cm.tab10(np.linspace(0, 0.9, len(scores)))
    bars   = ax.bar(names, scores, color=colors, alpha=0.85, edgecolor="white")
    for bar, s in zip(bars, scores):
        ax.text(bar.get_x()+bar.get_width()/2, s+0.001,
                f"{s:.4f}", ha="center", va="bottom", fontsize=8)
    ax.set_title(f"{label} — Validation Brier (v2)", fontsize=12)
    ax.set_ylabel("Brier (↓ better)")
    ax.set_ylim(0, hist["Baseline"]*1.05)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 11.2  Calibration curves ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
line_styles = ["-.", "--", ":", "-", (0,(3,1,1,1)), (0,(5,2))]
colors      = ["#5c85d6","#e08030","#2ca02c","#9467bd","#17becf","#bcbd22"]

for ax, y_val, bp, mv, label in [
    (axes[0], y_val_m, base_preds_m, val_meta_m, "Men's"),
    (axes[1], y_val_w, base_preds_w, val_meta_w, "Women's"),
]:
    for i, (name, probs) in enumerate(bp.items()):
        probs = np.clip(probs, PROB_MIN, PROB_MAX)
        frac, mean = calibration_curve(y_val, probs, n_bins=10)
        b = brier_score_loss(y_val, probs)
        ax.plot(mean, frac, marker="o", ms=4,
                ls=line_styles[i % len(line_styles)],
                color=colors[i % len(colors)],
                label=f"{name} ({b:.4f})")
    frac, mean = calibration_curve(y_val, mv, n_bins=10)
    b = brier_score_loss(y_val, mv)
    ax.plot(mean, frac, marker="o", ms=5, ls="-",
            color="#d62728", linewidth=2, label=f"★ Stack ({b:.4f})")
    ax.plot([0,1],[0,1],"k--",alpha=0.3,label="Perfect")
    ax.set_title(f"{label} — Calibration Curves (v2)", fontsize=11)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 11.3  Prediction distributions ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, probs, label, color in [
    (axes[0], sub_meta_m, "Men's",   "#1a73e8"),
    (axes[1], sub_meta_w, "Women's", "#e84393"),
]:
    ax.hist(probs, bins=40, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(0.5, color="black", ls="--", alpha=0.6, label="0.5")
    ax.axvline(probs.mean(), color="red", ls="-", alpha=0.7,
               label=f"Mean={probs.mean():.3f}")
    ax.set_title(f"{label} — Submission Distribution (v2)", fontsize=11)
    ax.set_xlabel("Predicted probability (Team1 wins)")
    ax.set_ylabel("Count"); ax.legend(); ax.grid(alpha=0.3)
    print(f"{label}: mean={probs.mean():.4f}  std={probs.std():.4f}  "
          f"range=[{probs.min():.3f},{probs.max():.3f}]")
plt.tight_layout()
plt.show()

## 12 · Save Submission

**v2: NO bias correction applied.** The submission means are already close to 0.5 naturally. Bias correction in v1 was computed from validation predictions but applied to submission predictions (different distributions), which moved means *further* from 0.5.

In [ ]:
from sklearn.isotonic import IsotonicRegression
from scipy.optimize import minimize_scalar

def fix_mens_calibration(val_preds, y_val, sub_preds):
    """
    Step 1: Isotonic regression fixes the shape (the 0.45→0.78 hump).
    Step 2: Temperature scaling optimizes the sharpness.
    Women's is untouched.
    """
    # ── Step 1: Isotonic ──────────────────────────────────────────────────
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(val_preds, y_val)
    cal_val = np.clip(iso.predict(val_preds), PROB_MIN, PROB_MAX)
    cal_sub = np.clip(iso.predict(sub_preds), PROB_MIN, PROB_MAX)

    b_before = brier_score_loss(y_val, np.clip(val_preds, PROB_MIN, PROB_MAX))
    b_after  = brier_score_loss(y_val, cal_val)
    print(f"  Isotonic: {b_before:.4f} → {b_after:.4f}  Δ={b_after-b_before:+.4f}")

    # ── Step 2: Temperature scaling on isotonic output ────────────────────
    def temp_scale(p, T):
        p = np.clip(p, 1e-7, 1-1e-7)
        return 1 / (1 + np.exp(-np.log(p/(1-p)) / T))

    res = minimize_scalar(
        lambda T: brier_score_loss(y_val,
            np.clip(temp_scale(cal_val, T), PROB_MIN, PROB_MAX)),
        bounds=(0.3, 2.0), method="bounded"
    )
    T_opt = res.x
    b_temp = brier_score_loss(y_val,
        np.clip(temp_scale(cal_val, T_opt), PROB_MIN, PROB_MAX))
    print(f"  TempScale T={T_opt:.4f}: {b_after:.4f} → {b_temp:.4f}  "
          f"Δ={b_temp-b_after:+.4f}  "
          f"({'sharpen' if T_opt<1 else 'soften'})")

    final_sub = np.clip(temp_scale(cal_sub, T_opt), PROB_MIN, PROB_MAX)
    print(f"  Final sub — mean={final_sub.mean():.4f}  std={final_sub.std():.4f}")
    return final_sub


In [ ]:
# ── No bias correction — means are already close to 0.5 ──────────────────────
# In Section 12, replace:
print("Calibrating Men's stack …")

#final_m = sub_meta_m.copy()
# With:
final_m = fix_mens_calibration(val_meta_m, y_val_m, sub_meta_m)

# Women's stays completely untouched:
final_w = sub_meta_w.copy()

print(f"Men's   mean before save: {final_m.mean():.4f}  std: {final_m.std():.4f}")
print(f"Women's mean before save: {final_w.mean():.4f}  std: {final_w.std():.4f}")
print(f"Targets: mean ≈ 0.490–0.510, std ≈ 0.28–0.35")

ids_m = sub_features[sub_features["IsWomens"]==0]["ID"].values
ids_w = sub_features[sub_features["IsWomens"]==1]["ID"].values

sub_df = pd.concat([
    pd.DataFrame({"ID": ids_m, "Pred": final_m}),
    pd.DataFrame({"ID": ids_w, "Pred": final_w}),
], ignore_index=True)

submission = sample_sub[["ID"]].merge(sub_df, on="ID", how="left")

nan_count = submission["Pred"].isna().sum()
if nan_count:
    print(f"⚠ {nan_count} NaN → filling with 0.5")
    submission["Pred"] = submission["Pred"].fillna(0.5)

submission["Pred"] = submission["Pred"].round(6)

assert list(submission.columns) == ["ID","Pred"]
assert submission["Pred"].between(0,1).all()
assert len(submission) == len(sample_sub), \
    f"Shape mismatch: {len(submission):,} vs {len(sample_sub):,}"

submission.to_csv(SUBMISSION_OUT, index=False)

print(f"\n✅ Saved : {SUBMISSION_OUT}")
print(f"   Shape  : {submission.shape}")
print(f"   Men's  mean : {final_m.mean():.4f}  (ideal ~0.500)")
print(f"   Women's mean: {final_w.mean():.4f}  (ideal ~0.500)")
print(f"   Overall mean: {submission['Pred'].mean():.4f}")
print(f"   Range  : [{submission['Pred'].min():.4f}, {submission['Pred'].max():.4f}]")

## 13 · Final Summary

In [ ]:
print("=" * 65)
print("  MARCH MADNESS 2026 — STACKED ENSEMBLE v2 RESULTS")
print("=" * 65)

for label, hist in [("Men's", hist_m), ("Women's", hist_w)]:
    print(f"\n  {label}")
    print(f"  {'─'*50}")
    base = hist["Baseline"]
    for key, name in [
        ("Baseline (0.5)_val", "Baseline (0.5)"),
        ("LR_val",             "  LR"),
        ("XGB (T-scaled)_val", "  XGBoost (T-scaled)"),
        ("NN_val",             "  NN"),
        ("FGN_val",            "  FGN (men only)"),
        ("VSN_val",            "  VSN (women only)"),
        ("Simple average_val", "  Simple avg"),
        ("★ Meta stack_val",   "  ★ Meta stack"),
    ]:
        if key not in hist: continue
        b   = hist[key]
        imp = (base - b)/base*100
        mrk = "  ←" if "Meta" in key else ""
        print(f"    {name:<25s}: {b:.4f}  ({imp:+.1f}%){mrk}")

print()
print(f"  Submission : {SUBMISSION_OUT}")
print(f"  Shape      : {submission.shape}")
print()
print("  v2 changes vs v1:")
print("    ✓ Bias correction REMOVED")
print("    ✓ XGBoost tighter (depth=2, min_child=5, alpha=0.5)")
print("    ✓ NN stronger regularization (drop 0.5/0.4, wd=0.05, patience=15)")
print("    ✓ Women's FGN → VSN (per-sample feature gating)")
print("    ✓ Men's meta: LR → GradientBoosting (captures model interactions)")
print("    ✓ SeedGap + IsUpsetter features added")
print("    ✓ Regular season games added to training (weight=0.1)")
print("=" * 65)